# VADER

Vader abstracts away the natural language processing and technological decision-making to democratise sentiment analysis. In this write-up, I will explore several decisions they made, assess the tool through a critical lens, and document my explorations of through a series of experiments.

#### Introduction to VADER *(from DH class):*
Sentiment analysis with VADER
* How positive/negative is this word?
* Contains ~7,500 words and phrases, each with a +/- score.
* Labelled by human volunteers

SCORING
The four scores it returns:

  #neg  - How much negative emotion it detected (0 to 1)
  #neu  - How much neutral language it detected (0 to 1)
  #pos  - How much positive emotion it detected (0 to 1)

  #compound - The overall score, from -1 (most negative) to +1 (most positive)
              #Think of this as VADER's single summary judgment of the whole text.
              #Above +0.05 = broadly positive
              #Below -0.05 = broadly negative
              #In between  = neutral

In this exercise, we ran the code on the Enron files, and have some preformed ideas of what to expect. As it is a public scandal, scores will likely tend towards the negative.

In [1]:
import nltk # Natural language toolkit
nltk.download('vader_lexicon') # VADER's emotional dictionary.  contains ~7,500 words and phrases,each with a positivity/negativity score.
nltk.download('punkt') # tokeniser

# engine:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
sid = SentimentIntensityAnalyzer() 

message_text = '''Like you, I am getting very frustrated with this process. I am genuinely trying to be as reasonable as possible. I am not trying to "hold up" the deal at the last minute. I'm afraid that I am being asked to take a fairly large leap of faith after this company (I don't mean the two of you -- I mean Enron) has screwed me and the people who work for me.'''

print(message_text)

scores = sid.polarity_scores(message_text)

# OUTPUT
# for each label in scores
for key in sorted(scores):
        # print label: score
        print('{0}: {1}, '.format(key, scores[key]), end='')

Like you, I am getting very frustrated with this process. I am genuinely trying to be as reasonable as possible. I am not trying to "hold up" the deal at the last minute. I'm afraid that I am being asked to take a fairly large leap of faith after this company (I don't mean the two of you -- I mean Enron) has screwed me and the people who work for me.
compound: -0.3804, neg: 0.093, neu: 0.836, pos: 0.071, 

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/urvashibalasubramaniam/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/urvashibalasubramaniam/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Interestingly, though the compound score is negative, the neutral score is the highest of the three individual metrics! Majorly, this text seems to contains neutral sentiment words/language. Why is this?

### Investigating high neutrality score:

To find out, I did some digging into how these scores are calculated and found an interesting insight on how the sentiment lexicon is constructed:
> All the lexical features were rated for the polarity and intensity on a scale from “-4: Extremely Negative” to “+4 Extremely Positive” by 10 independent human raters. 

*10* independent human raters is an interesting claim for generalisation, but I will explore that later in a different section. Let's focus on Neutrality for now.

> The average score is then used as the sentiment indicator for each lexical feature in the dictionary. For example, in Vader, the word “okay” has a positive rating of 0.9, “good” is 1.9 and “great” is 3.1, whereas “horrible” is -2.5, the frowning emoticon “:(“ is -2.2, and “sucks” is -1.5. Vader’s lexicon dictionary contains around 7,500 sentiment features in total and **any word not listed in the dictionary will be scored as “0: Neutral”**.

[Source: Ying Ma, Medium](https://medium.com/@mystery0116/nlp-how-does-nltk-vader-calculate-sentiment-6c32d0f5046b)

*Any words not listed in the dictionary are considered neutral!* This may cause biases in sentiment interpretation, as most words are not actually documented in this library. A very negative or positive text using *rarer* or unused words may go under the radar.

Also, compound score is calculated independent of the three metrics (neg, neu, pos)! [Source: StackOverflow, VADER source code](https://stackoverflow.com/questions/40325980/how-is-the-vader-compound-polarity-score-calculated-in-python-nltk). This explains the differing scores.


Now that we've solved that mystery, it opens up further questions about vocabulary coverage.

### How many words exist in various dictionaries documenting the human language? What percentage of English vocabulary is captured in this sentiment analysis library?

Here is the list of dictionaries and their headwords (the bolded section, ie "run" is the headword for "running", "ran" etc.)

| Dictionary | Headwords |
|---|---:|
| English Wiktionary | 923,306 |
| Collins English Dictionary | 732,000 |
| Oxford English Dictionary | 520,000 |
| Webster’s Third New International Dictionary and Addenda Section | 470,000 |
| The American Heritage Dictionary of the English Language (Fifth Edition) | 200,000 |
| WordNet 3.1 | 155,327 |

[Source: Wikipedia](https://en.wikipedia.org/wiki/List_of_dictionaries_by_number_of_words)

What does this mean for vocabulary coverage? Let's take the approximate 7500 words here as VADER's metric. In doing so, we can use a heuristic to estimate how much vocabulary is covered.

A caveat of this method is that these dictionaries serve different purposes and will make different decisions about exclusions and inclusions. For example, VADER includes modern and internet language that may be overlooked by some more traditional dictionaries. Thus, use this metric only for vocabulary volume comparison.

> The lexicon approach means that this algorithm constructed a dictionary that contains a comprehensive list of sentiment features. This lexical dictionary does not only contain words, but also phrases (such as “bad ass” and “the bomb”), emoticons such as :-) and sentiment-laden acronyms (such as “ROFL” and “WTF”). 

[Source: Ying Ma, Medium](https://medium.com/@mystery0116/nlp-how-does-nltk-vader-calculate-sentiment-6c32d0f5046b)

| Dictionary | Headwords | Volume Percentage |
|---|---:|---:|
| English Wiktionary | 923,306 | 0.8123% |
| Collins English Dictionary | 732,000 | 1.0246% |
| Oxford English Dictionary | 520,000 | 1.4423% |
| Webster’s Third New International Dictionary and Addenda Section | 470,000 | 1.5957% |
| The American Heritage Dictionary of the English Language (Fifth Edition) | 200,000 | 3.7500% |
| WordNet 3.1 | 155,327 | 4.8285% |

Using: `Volume Percentage = (7500 / Headwords) * 100`.

It's evident though that this vocabulary coverage is grossly insufficient. Can we analyse the world's texts with 0.8-4.8% of the target language's vocabulary?

This requires some statistical and linguistic analysis.

### Asking the linguists: how many words is enough for fluency?
If we treat VADER like an entity capable of word analysis with a vocabulary store, we can analogously look at human vocabularies and understand how many words are required to attain *fluency* in a language.

Here's an excerpt of a superbly interesting piece that tackles just that:
> So which words should we learn? Prof Webb says the most effective way to be able to speak a language quickly is to pick the 800 to 1,000 lemmas which appear most frequently in a language, and learn those. If you learn only 800 of the most frequently-used lemmas in English, you'll be able to understand 75% of the language as it is spoken in normal life. These 800 lemmas are a lot more valuable than other words, simply because they're used far more often. 

The analogy breaks in that humans are capable of contextually-analysing what a word might mean (as are many NLP tools like skipgrams, cbow, transformers), but VADER doesn't attempt to do that. It just leaves the sentiments unlabelled, which is a safer choice but ultimately less helpful.

### Experiment: How does VADER's vocabulary compare to English's most frequent words?

[Google NGram Frequency Dataset on Kaggle](https://www.kaggle.com/datasets/wheelercode/english-word-frequency-list)

VADER was a human-volunteer-based, and something is better than nothing, so it's best not to judge it too harshly. It is, however, not generalisable to the entire English language as a sentiment analysis tool, even setting dialect and word usage habits aside.

Other interesting insights from this article points to the inclusion of modifier words and grammar affecting scores via heuristics!
> Besides the sentiment lexicons, there are structures that are neutral inherently but can change the polarity of sentiment (such as “not” and “but”) or modify the intensity of the entire sentence (such as “very” and “extremely”). In Vader, the developers incorporated several heuristic rules that handles the cases of punctuation, capitalization, adverbs and contrastive conjunctions. Below are a few examples of how the degree modifiers boosted the positivity in the compound score of a sentence.

> Based on the heuristic rules and the normalization calculation, we can tell **Vader will average out the sentiment if the input text is relatively long** or has **several transition in term of tones and sentiment**. 

### Who decided the words that were included/excluded, and how?


The team began with established sentiment word banks, specifically LIWC (Linguistic Inquiry and Word Count), ANEW (Affective Norms for English Words), and GI (General Inquirer). The list was expanded to include features common in microblogs, such as emoticons, acronyms, initialisms, slang like LOL, WTF, nah, meh, and giggly.

As it was designed for microblogs, it often does poorly on texts larger than 280 words and tends to neutralise sentiment beyond that. It might, for instance, do very well on tweets, but not so well on essays.

The landscape of "social media" and its definition was so different than today that it would be difficult to retrofit it on today's posting habits. However, as it continues to be used today as a popular tool for "social media text analysis", it's worth investigating whether that prevalence is earned or just a remnant of what it used to be. 

ngram_freq.csv is not uploaded to this repository due to storage requirements but can be accessed at https://www.kaggle.com/datasets/wheelercode/english-word-frequency-list/data

In [2]:
# Compare with Google NGram Frequency Dataset on Kaggle
# load data
import pandas as pd
ngram = pd.read_csv('ngram_freq.csv')

In [3]:
# Construct a sentence with only the top 800 words from ngram_freq.csv
words = (
    ngram['word']
    .dropna()            # remove NaN
    .astype(str)         # ensure all are strings
    .str.strip()         # trim whitespace
)
words = words[words != ""]  # remove empty strings

top_800_words = words.head(800)   # ngram is already ordered
sentence = ' '.join(top_800_words.tolist())
print(sentence)

# Analyze the sentiment of the constructed sentence
sentence_scores = sid.polarity_scores(sentence)
print("\nSentiment scores for the constructed sentence:")
for key in sorted(sentence_scores):
    print('{0}: {1}, '.format(key, sentence_scores[key]), end='')

the of and to in a is that for as by be it with on was or not this are i from at he which an have his but you we all were they one had has will their been other if can may there would no more new such its when any these who so her time than do she some what about state only two into also out them our said under first my him up see made should after shall your most could then over each year work states where use years me between those same now many through upon s must well very general did before used because being part like united people during public section how number act even make court case system much three both per water law life day company service good way without order great american p while man however york long high b c us national right does city government information just within school against found power here world own another data committee business given know program little since present every house men report university following less back place total large take depar

We still have neutral words, which means they're not in the VADER library!
A lot of these, however, are unlikely to be included in the library in the first place. We accidentally get to our end goal because *the most common words  tend to be neutral anyway*. They are the stop words that we generally remove as they don't contribute sentiment!

What if we expand the vocabulary a bit to the top 5000 words?

In [7]:
# Construct a sentence with only the top 5000 words from ngram_freq.csv
words = (
    ngram['word']
    .dropna()            # remove NaN
    .astype(str)         # ensure all are strings
    .str.strip()         # trim whitespace
)
words = words[words != ""]  # remove empty strings

top_5000_words = words.head(5000)   # ngram is already ordered
sentence = ' '.join(top_5000_words.tolist())
print(sentence)

# Analyze the sentiment of the constructed sentence
sentence_scores = sid.polarity_scores(sentence)
print("\nSentiment scores for the constructed sentence:")
for key in sorted(sentence_scores):
    print('{0}: {1}, '.format(key, sentence_scores[key]), end='')

the of and to in a is that for as by be it with on was or not this are i from at he which an have his but you we all were they one had has will their been other if can may there would no more new such its when any these who so her time than do she some what about state only two into also out them our said under first my him up see made should after shall your most could then over each year work states where use years me between those same now many through upon s must well very general did before used because being part like united people during public section how number act even make court case system much three both per water law life day company service good way without order great american p while man however york long high b c us national right does city government information just within school against found power here world own another data committee business given know program little since present every house men report university following less back place total large take depar

Neutrality comes down by ~0.1. Our result still holds.

#### Who were the volunteers here? Where were they from?
Linguistics and sentiment analysis, on a human scale, is highly sociocultural. Therefore, if we are promoting VADER as a global tool to assess sentiment in language, we must pay attention to *whose* language sensibilities and sentiments this library is capturing. Which brings me to this very important question.

I managed to obtain the list of contributors to the VADER project, publicly posted and downloadable as a dataset "names"

It would be very easy to use a name-identification library to classify female/male ratio contributions and probabilistic nationalities, but I wrote the entirety of this codeblock and had a nagging feeling in my stomach as I did. Though names encode a lot of cultural information about location and demographics, unearthing these in bulk computationally feels unethical and misleading. So I had completed this section but have since deleted it. 

If on further discussion, we find that this process can be done ethically, then I still have the code used to produce the contributor demographics (male/female ratio and nationality). For now, from what I've seen, names can be misleading in data. Thus, we move on to the next section.

### Experiments with the tool

>Performance Validation of the algorithm also attested that Vader performs exceptionally well in the social media domain, and outperforms human raters at classifying the sentiment of tweets.

>Because of the embedded lexicon and rules, Vader is computationally economical especially comparing to the machine learning algorithms that requires massive operation for word embedding and training. Even though the sentiment features are restricted within the built-in lexicon and rules, it is relatively easy to modify and extend the sentimental vocabulary and tailored the Vader to specific contextual use cases.

The idea here is that VADER is good for a lightweight model, and its shortcomings are acknowleged.

## Experiment: Confusing VADER
I wanted to run it on difficult-to-classify messages, with mixed sentiment, sarcasm, double negatives, ambiguity and Indian-dialect sentences.

In [19]:
# for SENTENCE-level analysis
import nltk
nltk.download('punkt_tab')
import nltk.data
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk import sentiment
from nltk import word_tokenize

# Next, we initialize VADER so we can use it within our Python script
sid = SentimentIntensityAnalyzer()

# We will also initialize our 'english.pickle' function and give it a short name
#A "tokenizer" is a tool that breaks text into smaller units —  in this case, sentences.
   #"punkt" is the name of the algorithm it uses to decide where one sentence ends and another begins.
   #(It looks for full stops, capital letters, and other signals.)

tokenizer = nltk.data.load('tokenizers/punkt/english.pickle')

message_text = '''"Oh, fantastic! Another 4-hour meeting that definitely couldn’t have been an email. The plot was slow and characters were one-dimensional, but the cinematography was breathtaking." "It's not that I don't love the product, but it certainly isn't the best in its class." "If you are looking for a phone with terrible battery life, poor camera, and fragile screen, this is the one for you." "This country is going to the dogs." "The service was okay, but the food was mediocre at best." "I can't believe how amazing this movie was! The acting, plot, and special effects were all top-notch." "This restaurant is a hidden gem. The food is delicious and the atmosphere is cozy." "I had a terrible experience at this hotel. The room was dirty and the staff was rude." "The new phone has a sleek design and impressive features, but the battery life is disappointing." "I am so grateful for my supportive friends and family. They always have my back. You're so skibidi toilet ballerina cappucina. Caused a motherquake on the cunt scale."'''

# The tokenize method breaks up the paragraph into a list of strings. This turns our paragraph into a list of individual sentences,

sentences = tokenizer.tokenize(message_text)

# We add the additional step of iterating through the list of sentences and calculating and printing polarity scores for each one.

for sentence in sentences:
        print(sentence)
        scores = sid.polarity_scores(sentence)
        for key in sorted(scores):
                print('{0}: {1}, '.format(key, scores[key]), end='')
        print()

"Oh, fantastic!
compound: 0.5983, neg: 0.0, neu: 0.204, pos: 0.796, 
Another 4-hour meeting that definitely couldn’t have been an email.
compound: 0.4019, neg: 0.0, neu: 0.769, pos: 0.231, 
The plot was slow and characters were one-dimensional, but the cinematography was breathtaking."
compound: 0.0, neg: 0.0, neu: 1.0, pos: 0.0, 
"It's not that I don't love the product, but it certainly isn't the best in its class."
compound: -0.1471, neg: 0.202, neu: 0.577, pos: 0.221, 
"If you are looking for a phone with terrible battery life, poor camera, and fragile screen, this is the one for you."
compound: -0.7351, neg: 0.246, neu: 0.754, pos: 0.0, 
"This country is going to the dogs."
compound: 0.0, neg: 0.0, neu: 1.0, pos: 0.0, 
"The service was okay, but the food was mediocre at best."
compound: 0.1154, neg: 0.0, neu: 0.873, pos: 0.127, 
"I can't believe how amazing this movie was!
compound: -0.521, neg: 0.325, neu: 0.675, pos: 0.0, 
The acting, plot, and special effects were all top-notch.

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/urvashibalasubramaniam/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


VADER evidently:
* Can't deal with sarcasm
* Can't deal with explicitly non-negative words. A more complicated NLP tool that uses embeddings would be better for this, because single words don't often hold all the meaning. It is the relationships and implicit meanings held in sentences that reveal sentiment.
* Can detect mixed emotion decently well in the tested example.
* Okay, in most articles I read about VADER, is ranked at 0.9 (positive!) while *okay* the word holds much more depth and meaning, and is often not used in a positive context. It is these subtleties that are lost on VADER.
* Negations are conflated with negativity. "I can't belivee how amazing this movie was!" got a compound *negative* score, even though it's overwhelingly positive, because the "can't" dragged down the score.
* Can't handle phrases and idioms well, because they are phrases and don't operate at the single-word-resolution. "They always have my back" was neutral. "This country is going to the dogs" is also perfectly neutral, even though these statements hold significant amounts of emotion.

## Indian Dialect Handling
I had some fun coming up with indian-ism sentences that uniquely reflect the Indian dialect. How does it handle them, with both readability and sentiment?

In [15]:
# for SENTENCE-level analysis
import nltk
nltk.download('punkt_tab')
import nltk.data
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk import sentiment
from nltk import word_tokenize

# Next, we initialize VADER so we can use it within our Python script
sid = SentimentIntensityAnalyzer()

# We will also initialize our 'english.pickle' function and give it a short name
#A "tokenizer" is a tool that breaks text into smaller units —  in this case, sentences.
   #"punkt" is the name of the algorithm it uses to decide where one sentence ends and another begins.
   #(It looks for full stops, capital letters, and other signals.)

tokenizer = nltk.data.load('tokenizers/punkt/english.pickle')

message_text = 'Kindly do the needful. What is your good name? Can we prepone the meeting? Inshallah she will pass out of college this year. He is my real brother. Do one thing. You just sit on my head only. I am going by walk. Yes, I told you, no? Why are you eating my brain? That only.'

# The tokenize method breaks up the paragraph into a list of strings. This turns our paragraph into a list of individual sentences,

sentences = tokenizer.tokenize(message_text)

# We add the additional step of iterating through the list of sentences and calculating and printing polarity scores for each one.

for sentence in sentences:
        print(sentence)
        scores = sid.polarity_scores(sentence)
        for key in sorted(scores):
                print('{0}: {1}, '.format(key, scores[key]), end='')
        print()

Kindly do the needful.
compound: 0.4939, neg: 0.0, neu: 0.484, pos: 0.516, 
What is your good name?
compound: 0.4404, neg: 0.0, neu: 0.58, pos: 0.42, 
Can we prepone the meeting?
compound: 0.0, neg: 0.0, neu: 1.0, pos: 0.0, 
Inshallah she will pass out of college this year.
compound: 0.0, neg: 0.0, neu: 1.0, pos: 0.0, 
He is my real brother.
compound: 0.0, neg: 0.0, neu: 1.0, pos: 0.0, 
Do one thing.
compound: 0.0, neg: 0.0, neu: 1.0, pos: 0.0, 
You just sit on my head only.
compound: 0.0, neg: 0.0, neu: 1.0, pos: 0.0, 
I am going by walk.
compound: 0.0, neg: 0.0, neu: 1.0, pos: 0.0, 
Yes, I told you, no?
compound: 0.128, neg: 0.319, neu: 0.29, pos: 0.391, 
Why are you eating my brain?
compound: 0.0, neg: 0.0, neu: 1.0, pos: 0.0, 
That only.
compound: 0.0, neg: 0.0, neu: 1.0, pos: 0.0, 


[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/urvashibalasubramaniam/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


* "Kindly" bumped up the first sentence's score, even though it's generally a neutral statement. This categorisation isn't explicitly wrong, though.
* "Good" also bumped up the positive score for someone asking for someone's legal name, a usually neutral statement.
* Prepone was unrecognised, and this surprisingly got a completely neutral score. It is not emotionally charged though, so again this isn't technically wrong.
* Unsurprisingly, most of the sentences, even when emotionally charged, got a neutral zero score.

In [16]:
#Readability
#Readability analysis measures how **easy or difficult** a text is to read, and produces a score — usually expressed as a **school grade level** or a number on a scale.
#It does this by measuring things like:
#- **How long are the sentences?** (longer = harder)
#- **How many syllables do the words have?** (more syllables = harder)
#- **How many long or rare words are used?**

#The most common readability measures are:

# **Flesch Reading Ease** | Overall ease, 0–100 | Higher = easier. 60–70 = plain English. Below 30 = very difficult. |
# **Flesch-Kincaid Grade** | US school grade level | 8 = readable by an 8th grader. 16 = university level. |
#**Gunning Fog Index** | Years of education needed | Below 12 = accessible. Above 17 = very academic. |
#**SMOG Index** | Years of schooling to understand |


# This cell stores our Dickens passage in a variable called sample_text.
# Every analysis below will refer back to this same variable.
# Run this cell first before anything else.

sample_text = """It was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions, though not quickly enough to prevent a swirl of gritty dust from entering along with him. 

The hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide: the face of a man of about forty-five, with a heavy black moustache and ruggedly handsome features. Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working, and at present the electric current was cut off during daylight hours. It was part of the economy drive in preparation for Hate Week. The flat was seven flights up, and Winston, who was thirty-nine and had a varicose ulcer above his right ankle, went slowly, resting several times on the way. On each landing, opposite the lift-shaft, the poster with the enormous face gazed from the wall. It was one of those pictures which are so contrived that the eyes follow you about when you move. BIG BROTHER IS WATCHING YOU, the caption beneath it ran. 

Inside the flat a fruity voice was reading out a list of figures which had something to do with the production of pig-iron. The voice came from an oblong metal plaque like a dulled mirror which formed part of the surface of the right-hand wall. Winston turned a switch and the voice sank somewhat, though the words were still distinguishable. The instrument (the telescreen, it was called) could be dimmed, but there was no way of shutting it off completely. He moved over to the window: a smallish, frail figure, the meagreness of his body merely emphasized by the blue overalls which were the uniform of the party. His hair was very fair, his face naturally sanguine, his skin roughened by coarse soap and blunt razor blades and the cold of the winter that had just ended. 

Outside, even through the shut window-pane, the world looked cold. Down in the street little eddies of wind were whirling dust and torn paper into spirals, and though the sun was shining and the sky a harsh blue, there seemed to be no colour in anything, except the posters that were plastered everywhere. The blackmoustachio'd face gazed down from every commanding corner. There was one on the house-front immediately opposite. BIG BROTHER IS WATCHING YOU, the caption said, while the dark eyes looked deep into Winston's own. Down at streetlevel another poster, torn at one corner, flapped fitfully in the wind, alternately covering and uncovering the single word INGSOC. In the far distance a helicopter skimmed down between the roofs, hovered for an instant like a bluebottle, and darted away again with a curving flight. It was the police patrol, snooping into people's windows. The patrols did not matter, however. Only the Thought Police mattered. 

Behind Winston's back the voice from the telescreen was still babbling away about pig-iron and the overfulfilment of the Ninth Three-Year Plan. The telescreen received and transmitted simultaneously. Any sound that Winston made, above the level of a very low whisper, would be picked up by it, moreover, so long as he remained within the field of vision which the metal plaque commanded, he could be seen as well as heard. There was of course no way of knowing whether you were being watched at any given moment. How often, or on what system, the Thought Police plugged in on any individual wire was guesswork. It was even conceivable that they watched everybody all the time. But at any rate they could plug in your wire whenever they wanted to. You had to live -- did live, from habit that became instinct -- in the assumption that every sound you made was overheard, and, except in darkness, every movement scrutinized. 

Winston kept his back turned to the telescreen. It was safer, though, as he well knew, even a back can be revealing. A kilometre away the Ministry of Truth, his place of work, towered vast and white above the grimy landscape. This, he thought with a sort of vague distaste -- this was London, chief city of Airstrip One, itself the third most populous of the provinces of Oceania. He tried to squeeze out some childhood memory that should tell him whether London had always been quite like this. Were there always these vistas of rotting nineteenth-century houses, their sides shored up with baulks of timber, their windows patched with cardboard and their roofs with corrugated iron, their crazy garden walls sagging in all directions? And the bombed sites where the plaster dust swirled in the air and the willow-herb straggled over the heaps of rubble; and the places where the bombs had cleared a larger patch and there had sprung up sordid colonies of wooden dwellings like chicken-houses? But it was no use, he could not remember: nothing remained of his childhood except a series of bright-lit tableaux occurring against no background and mostly unintelligible. 

The Ministry of Truth -- Minitrue, in Newspeak -- was startlingly different from any other object in sight. It was an enormous pyramidal structure of glittering white concrete, soaring up, terrace after terrace, 300 metres into the air. From where Winston stood it was just possible to read, picked out on its white face in elegant lettering, the three slogans of the Party: """

print("Text loaded successfully! Here it is:")
print("="*60)
print(sample_text)

# First we install the textstat library — a ready-made readability toolkit.
# 'pip install' means: go and download this toolbox from the internet.
# You only need to do this once per session.

!pip install textstat --quiet
# Now we import textstat so we can use it
import textstat

# --- RUNNING THE READABILITY SCORES ---
# Each line below calls a different readability formula on our text.
# Think of each formula as a different critic applying their own rubric.

flesch_ease    = textstat.flesch_reading_ease(sample_text)
flesch_grade   = textstat.flesch_kincaid_grade(sample_text)
fog            = textstat.gunning_fog(sample_text)
smog           = textstat.smog_index(sample_text)
avg_sent_len   = textstat.avg_sentence_length(sample_text)
avg_syllables  = textstat.avg_syllables_per_word(sample_text)
word_count     = textstat.lexicon_count(sample_text)
sentence_count = textstat.sentence_count(sample_text)

# --- PRINTING THE RESULTS ---
# The '\n' you see below just means 'start a new line' — it's invisible spacing.

print("READABILITY ANALYSIS: A Tale of Two Cities (Opening)")
print("=" * 55)
print(f"\n📊 BASIC TEXT STATISTICS")
print(f"   Word count               : {word_count}")
print(f"   Sentence count           : {sentence_count}")
print(f"   Avg words per sentence   : {avg_sent_len}")
print(f"   Avg syllables per word   : {avg_syllables}")

print(f"\n📈 READABILITY SCORES")
print(f"   Flesch Reading Ease      : {flesch_ease}")
print(f"   Flesch-Kincaid Grade     : {flesch_grade}  (US school grade level)")
print(f"   Gunning Fog Index        : {fog}")
print(f"   SMOG Index               : {smog}")

print(f"\n📖 WHAT THIS MEANS")
if flesch_ease >= 60:
    print(f"   Flesch Ease of {flesch_ease}: Fairly easy to read — plain language.")
elif flesch_ease >= 30:
    print(f"   Flesch Ease of {flesch_ease}: Moderately difficult — academic or literary.")
else:
    print(f"   Flesch Ease of {flesch_ease}: Very difficult — dense, complex prose.")

Text loaded successfully! Here it is:
It was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions, though not quickly enough to prevent a swirl of gritty dust from entering along with him. 

The hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide: the face of a man of about forty-five, with a heavy black moustache and ruggedly handsome features. Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working, and at present the electric current was cut off during daylight hours. It was part of the economy drive in preparation for Hate Week. The flat was seven flights up, and Winston, who was thirty-nine and had a varicose ulcer above h

/var/folders/7s/fdc7zcc527q_6118s727lxl80000gn/T/ipykernel_3420/1483967553.py:54: DeprecationWarning: The 'avg_sentence_length' method has been deprecated due to being the same as 'words_per_sentence'. This method will be removed in thefuture.
  avg_sent_len   = textstat.avg_sentence_length(sample_text)


Orwell's 1984 got a "fairly easy to read" score. This is, of course, a heuristic. The point is that difficult ideas can be presented in plain language. At a word-level, this seems to do a good job. Whether the heuristic is useful for generally understanding and representing "readability" is another, more complicated question.

In [17]:
#  Word Frequency
## What Words Does This Text Obsess Over?

### What is Word Frequency Analysis?

#This is perhaps the simplest analysis — we simply **count how many times each word appears**.

# We need NLTK for this analysis — it contains the stopwords list
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from collections import Counter
import string

print("Libraries loaded successfully.")
# --- STEP 1: TOKENISE ---
# 'Tokenising' means breaking the text into individual words.
# .lower() converts everything to lowercase so 'Times' and 'times' count as the same word.

tokens = word_tokenize(sample_text.lower())

print(f"Total tokens (words + punctuation): {len(tokens)}")
print(f"First 10 tokens: {tokens[:10]}")
# --- STEP 2: REMOVE STOPWORDS AND PUNCTUATION ---
# Stopwords are words like 'the', 'of', 'it', 'was' — common but meaningless.
# Removing them lets us see the words that actually carry meaning.

stop_words = set(stopwords.words('english'))

# We keep a word only if:
#   - it is NOT a stopword
#   - it is NOT a punctuation mark
filtered_words = [
    word for word in tokens
    if word not in stop_words
    and word not in string.punctuation
    and word.isalpha()  # isalpha() means: only keep actual words, not numbers or symbols
]

print(f"Words after removing stopwords and punctuation: {len(filtered_words)}")
print(f"\nStopwords removed include words like: {list(stop_words)[:10]}")
# --- STEP 3: COUNT AND DISPLAY ---
# Counter counts how many times each word appears.
# .most_common(15) gives us the top 15.

word_freq = Counter(filtered_words)
top_words = word_freq.most_common(15)

print("WORD FREQUENCY ANALYSIS: A Tale of Two Cities (Opening)")
print("=" * 50)
print(f"\nTop 15 most frequent meaningful words:\n")
print(f"{'Rank':<6} {'Word':<20} {'Count':<8} {'Visual Bar'}")
print("-" * 55)

for rank, (word, count) in enumerate(top_words, 1):
    bar = '█' * (count * 3)  # Each occurrence = 3 blocks, for a visual effect
    print(f"{rank:<6} {word:<20} {count:<8} {bar}")

Libraries loaded successfully.
Total tokens (words + punctuation): 1066
First 10 tokens: ['it', 'was', 'a', 'bright', 'cold', 'day', 'in', 'april', ',', 'and']
Words after removing stopwords and punctuation: 476

Stopwords removed include words like: ['through', 'himself', 'itself', "hadn't", 'again', 'other', 'where', "we'll", 'm', 'ours']
WORD FREQUENCY ANALYSIS: A Tale of Two Cities (Opening)

Top 15 most frequent meaningful words:

Rank   Word                 Count    Visual Bar
-------------------------------------------------------
1      winston              9        ███████████████████████████
2      face                 6        ██████████████████
3      one                  5        ███████████████
4      though               4        ████████████
5      even                 4        ████████████
6      voice                4        ████████████
7      like                 4        ████████████
8      telescreen           4        ████████████
9      could                4   

This gives us a good idea of the theme and style of the writing, so frequency of words is still a useful metric.

Takeaways on VADER (2014)

I was happy to scroll through the list of contributors and see a decently diverse set of names, even if I chose not to quantify this. As interesting as it is as an open-source project, I don't think we should brush past its many shortcomings.
The low vocabulary coverage was surprising, for such a widespread tool. Statistically and linguistically, we can manage with smaller datasets and high frequency words. However, since these high frequency words are mostly *neutral*, I'm unsure of how they went about selecting VADER's vocabulary.

VADER is a lightweight tool computationally, but it need not be one functionally. Many of the problems it faces, for instance, only processing at the word-level and not at the sentence level, and the inability for co-occuring words or bigrams to affect each other semantically is problematic (contextualisation). However, these issues have largely been resolved in advanced NLP with tools like transformers. Words carry both sentimental and syntactical meaning, and are represented as embeddings. Neighbouring words may affect each other's embeddings (and thus their meaning) mathematically and in-practice. 

However, VADER cannot handle this. It is still widespread and advertised as a "democratised tool". However, this democratisation is of a technology that doesn't match up to its non-democratised counterparts. It can be deeply misleading in its misclassification, especially to people who do not understand its workings and may take its output for granted.

Online supporters of VADERS often praise its high performance and widespread use in *social media*, but if my simple cursory experiments were able to produce so many false positives and false negatives, I find it insufficient as an NLP tool for social media text. Social media is by definition global, ridden with slang and meaning that isn't immediately obvious. Popularity and virality, according to meme theory, depends heavily on *novelty*. In fact, social media is often the source of new types and structures of language. I actually find it to be the *hardest* area to study with NLP, purely because of the evolving linguistics, socialects and often absurd meaning-making systems that depend heavily on references to other popular media and memes lost over time.

If we are to understand reviews, social media and general human behaviour through computational tools, and if we are to democratise them for ease of use and computational efficiency, then we must do it well. I don't think the fact that it is open-source and lightweight is of any use if the tool itself is misleading and unhelpful. I don't think it's a good idea to hold democratised tools to a lower standard. If anything, these should be made *less* prone to bias because this bias is likely to go undetected if the NLP process is abstracted away.

That being said, it can perform some basic functions well. It often gets confused by the presence of "can't" and "not" and pushes positive sentences into negative territory, even at the basic intended level. It doesn't account for multiple possible ratings and use cases of a single word, which is slighly ignorant of the fundamental functions of words in language. My biggest qualm was with its false negative of neutrality, which basically forgoes a huge amount of the sentiment analysis task if the words used are unrepresented in VADER's word data.

VADER came up in 2014, when statistical methods had been overtaken by transformers and learning for two years already. However, I may be retrofitting our rate of technological development on the past. For the time, VADER was probably a great and useful tool. Now, I hope more advanced libraries become open-source, and that good text analysis is not hidden behind propriety software and difficult learning curves for much longer.

In [20]:
import nltk # Natural language toolkit
nltk.download('vader_lexicon') # VADER's emotional dictionary.  contains ~7,500 words and phrases,each with a positivity/negativity score.
nltk.download('punkt') # tokeniser

# engine:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
sid = SentimentIntensityAnalyzer() 

message_text = '''Takeaways on VADER (2014)

I was happy to scroll through the list of contributors and see a decently diverse set of names, even if I chose not to quantify this. As interesting as it is as an open-source project, I don't think we should brush past its many shortcomings.
The low vocabulary coverage was surprising, for such a widespread tool. Statistically and linguistically, we can manage with smaller datasets and high frequency words. However, since these high frequency words are mostly *neutral*, I'm unsure of how they went about selecting VADER's vocabulary.

VADER is a lightweight tool computationally, but it need not be one functionally. Many of the problems it faces, for instance, only processing at the word-level and not at the sentence level, and the inability for co-occuring words or bigrams to affect each other semantically is problematic (contextualisation). However, these issues have largely been resolved in advanced NLP with tools like transformers. Words carry both sentimental and syntactical meaning, and are represented as embeddings. Neighbouring words may affect each other's embeddings (and thus their meaning) mathematically and in-practice. 

However, VADER cannot handle this. It is still widespread and advertised as a "democratised tool". However, this democratisation is of a technology that doesn't match up to its non-democratised counterparts. It can be deeply misleading in its misclassification, especially to people who do not understand its workings and may take its output for granted.

Online supporters of VADERS often praise its high performance and widespread use in *social media*, but if my simple cursory experiments were able to produce so many false positives and false negatives, I find it insufficient as an NLP tool for social media text. Social media is by definition global, ridden with slang and meaning that isn't immediately obvious. Popularity and virality, according to meme theory, depends heavily on *novelty*. In fact, social media is often the source of new types and structures of language. I actually find it to be the *hardest* area to study with NLP, purely because of the evolving linguistics, socialects and often absurd meaning-making systems that depend heavily on references to other popular media and memes lost over time.

If we are to understand reviews, social media and general human behaviour through computational tools, and if we are to democratise them for ease of use and computational efficiency, then we must do it well. I don't think the fact that it is open-source and lightweight is of any use if the tool itself is misleading and unhelpful. I don't think it's a good idea to hold democratised tools to a lower standard. If anything, these should be made *less* prone to bias because this bias is likely to go undetected if the NLP process is abstracted away.

That being said, it can perform some basic functions well. It often gets confused by the presence of "can't" and "not" and pushes positive sentences into negative territory, even at the basic intended level. It doesn't account for multiple possible ratings and use cases of a single word, which is slighly ignorant of the fundamental functions of words in language. My biggest qualm was with its false negative of neutrality, which basically forgoes a huge amount of the sentiment analysis task if the words used are unrepresented in VADER's word data.

VADER came up in 2014, when statistical methods had been overtaken by transformers and learning for two years already. However, I may be retrofitting our rate of technological development on the past. For the time, VADER was probably a great and useful tool. Now, I hope more advanced libraries become open-source, and that good text analysis is not hidden behind propriety software and difficult learning curves for much longer.'''

print(message_text)

scores = sid.polarity_scores(message_text)

# OUTPUT
# for each label in scores
for key in sorted(scores):
        # print label: score
        print('{0}: {1}, '.format(key, scores[key]), end='')

Takeaways on VADER (2014)

I was happy to scroll through the list of contributors and see a decently diverse set of names, even if I chose not to quantify this. As interesting as it is as an open-source project, I don't think we should brush past its many shortcomings.
The low vocabulary coverage was surprising, for such a widespread tool. Statistically and linguistically, we can manage with smaller datasets and high frequency words. However, since these high frequency words are mostly *neutral*, I'm unsure of how they went about selecting VADER's vocabulary.

VADER is a lightweight tool computationally, but it need not be one functionally. Many of the problems it faces, for instance, only processing at the word-level and not at the sentence level, and the inability for co-occuring words or bigrams to affect each other semantically is problematic (contextualisation). However, these issues have largely been resolved in advanced NLP with tools like transformers. Words carry both sentimen

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/urvashibalasubramaniam/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/urvashibalasubramaniam/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


*VADER thinks I really love this tool. Enough said!*

An exploration project by Urvashi Balasubramaniam. Digital Humanities 2026.